# Symbolic Pneumonia Classifier with PySR

10-fold stratified cross-validation of a closed-form symbolic classifier for binary pneumonia detection on paediatric chest radiographs.

**Inputs** (place in the working directory):
- `normal_train2.csv`, `bacteria_train2.csv`, `virus_train2.csv`
- `normal_test2.csv`,  `bacteria_test2.csv`,  `virus_test2.csv`

Each CSV contains the 10 radiomic features extracted by the MATLAB pipeline plus an image identifier.

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm             import SVC
from sklearn.metrics         import (accuracy_score, roc_auc_score, f1_score,
                                     confusion_matrix, ConfusionMatrixDisplay,
                                     roc_curve, auc)
from pysr import PySRRegressor

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
DATA_DIR = '.'
OUT_DIR  = '.'
N_SPLITS = 10

FEATURES = ['EntropyMean','EntropyMax','Entropy90','Haze','Solidity',
            'Contrast','Homogeneity','Skewness','Kurtosis','FractalDim']

PYSR_KWARGS = dict(
    niterations       = 40,
    binary_operators  = ['+', '-', '*', '/'],
    unary_operators   = ['log', 'sqrt'],
    populations       = 15,
    population_size   = 33,
    maxsize           = 25,
    parsimony         = 0.005,
    elementwise_loss  = 'L2DistLoss()',
    progress          = False,
    verbosity         = 0,
    random_state      = RANDOM_STATE,
    deterministic     = True,
    parallelism       = 'serial',
    temp_equation_file= True,
)

In [ ]:
def load_split(prefix):
    n = pd.read_csv(os.path.join(DATA_DIR, f'normal_{prefix}.csv')).dropna()
    b = pd.read_csv(os.path.join(DATA_DIR, f'bacteria_{prefix}.csv')).dropna()
    v = pd.read_csv(os.path.join(DATA_DIR, f'virus_{prefix}.csv')).dropna()
    n['Label']='Normal'; b['Label']='Bacteria'; v['Label']='Virus'
    return pd.concat([n, b, v], ignore_index=True)

df_train = load_split('train2')
df_test  = load_split('test2')

X_train_raw = df_train[FEATURES].values.astype(float)
X_test_raw  = df_test [FEATURES].values.astype(float)
y_train     = (df_train['Label'] != 'Normal').astype(int).values
y_test      = (df_test ['Label'] != 'Normal').astype(int).values

scaler  = StandardScaler().fit(X_train_raw)
X_train = scaler.transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f'Train: {df_train.shape[0]} images   Test: {df_test.shape[0]} images')

In [ ]:
def t95_ci(values):
    a = np.asarray(values, dtype=float)
    se = a.std(ddof=1) / np.sqrt(len(a))
    h  = se * stats.t.ppf(0.975, len(a) - 1)
    return float(a.mean()), float(h)

def binary_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return dict(
        acc  = accuracy_score(y_true, y_pred),
        auc  = roc_auc_score(y_true, y_score),
        sens = tp / (tp + fn) if (tp+fn) else 0.0,
        spec = tn / (tn + fp) if (tn+fp) else 0.0,
        f1   = f1_score(y_true, y_pred),
    )

## 10-fold stratified cross-validation with PySR

In [ ]:
skf  = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
rows, equations = [], []

for fold, (tr, te) in enumerate(skf.split(X_train_raw, y_train), start=1):
    scl = StandardScaler().fit(X_train_raw[tr])
    Xtr, Xte = scl.transform(X_train_raw[tr]), scl.transform(X_train_raw[te])
    ytr, yte = y_train[tr], y_train[te]

    t0 = time.time()
    model = PySRRegressor(**PYSR_KWARGS)
    model.fit(Xtr, ytr.astype(float), variable_names=FEATURES)

    y_score = np.asarray(model.predict(Xte), dtype=float)
    y_pred  = (y_score >= 0.5).astype(int)

    m = binary_metrics(yte, y_pred, y_score)
    m.update(fold=fold, time_s=round(time.time() - t0, 1))
    rows.append(m)

    best_eq = model.get_best()
    equations.append(dict(fold=fold,
                          complexity=int(best_eq.get('complexity', -1)),
                          loss=float(best_eq.get('loss', np.nan)),
                          equation=str(best_eq.get('equation',''))))

    print(f'[fold {fold:2d}/{N_SPLITS}]  acc={m["acc"]:.3f}  auc={m["auc"]:.3f}  '
          f'complexity={equations[-1]["complexity"]}  time={m["time_s"]:.1f}s')

df_cv = pd.DataFrame(rows)
df_eq = pd.DataFrame(equations)

In [ ]:
summary = {}
for met in ['acc','auc','sens','spec','f1']:
    mu, h = t95_ci(df_cv[met].values)
    summary[met] = f'{mu:.3f} \u00b1 {h:.3f}'
print('=== PySR symbolic classifier \u2014 10-fold CV (binary) ===')
for k, v in summary.items():
    print(f'  {k:5s}: {v}')

print('\n=== Equation discovered in each fold ===')
for _, r in df_eq.iterrows():
    print(f'  fold {int(r.fold):2d} | complexity {int(r.complexity):2d} | '
          f'loss {r.loss:.4f}\n    {r.equation}')

#df_cv.to_csv(os.path.join(OUT_DIR, 'cv10_pysr_raw.csv'),       index=False)
#df_eq.to_csv(os.path.join(OUT_DIR, 'cv10_pysr_equations.csv'), index=False)

## Held-out test set: SVM (RBF) vs Symbolic Classifier

The symbolic classifier is evaluated using the equation discovered in fold 8 (a representative compact form). The decision threshold is selected on the training set. We can compare with any black-box classifier we have in the paper. Here we show with SVM.

In [ ]:
svm = SVC(kernel='rbf', C=1, gamma=0.1, probability=True,
          class_weight='balanced').fit(X_train, y_train)
svm_proba = svm.predict_proba(X_test)[:, 1]
cm_svm = confusion_matrix(y_test, svm.predict(X_test))

def sc_score(X_scaled):
    em = X_scaled[:, FEATURES.index('EntropyMean')]
    fd = X_scaled[:, FEATURES.index('FractalDim')]
    arg = (em - 0.5513386 * fd) * -0.71133715 + 1.9655408
    return np.log(np.abs(arg))

z_train = sc_score(X_train)
z_test  = sc_score(X_test)

cands  = np.linspace(z_train.min(), z_train.max(), 401)
best_t = max(cands, key=lambda t: accuracy_score(y_train, (z_train > t).astype(int)))
sr_preds = (z_test > best_t).astype(int)
cm_sr    = confusion_matrix(y_test, sr_preds)

print(f'SVM  test acc: {accuracy_score(y_test, svm.predict(X_test)):.3f}   '
      f'AUC: {roc_auc_score(y_test, svm_proba):.3f}')
print(f'SC   test acc: {accuracy_score(y_test, sr_preds):.3f}   '
      f'AUC: {roc_auc_score(y_test, z_test):.3f}   threshold: {best_t:.3f}')

In [ ]:
def roc_pair(y_true, score_pos):
    fpr_p, tpr_p, _ = roc_curve(y_true,  score_pos, pos_label=1)
    fpr_n, tpr_n, _ = roc_curve(y_true, -score_pos, pos_label=0)
    return (fpr_p, tpr_p, auc(fpr_p, tpr_p)), (fpr_n, tpr_n, auc(fpr_n, tpr_n))

(svm_p, svm_n) = roc_pair(y_test, svm_proba)
(sc_p,  sc_n ) = roc_pair(y_test, z_test)

fig, ax = plt.subplots(2, 2, figsize=(12, 10))

ConfusionMatrixDisplay(confusion_matrix=cm_svm,
                       display_labels=['Normal','Pneumonia']
    ).plot(cmap='Blues', ax=ax[0,0], colorbar=False)
ax[0,0].set_title('(a) SVM (RBF)')

ax[0,1].plot(svm_n[0], svm_n[1], color='royalblue',
             label=f'Normal (AUC = {svm_n[2]:.2f})')
ax[0,1].plot(svm_p[0], svm_p[1], color='crimson',
             label=f'Pneumonia (AUC = {svm_p[2]:.2f})')
ax[0,1].plot([0,1],[0,1],'--', color='black', lw=1)
ax[0,1].set_xlabel('False Positive Rate'); ax[0,1].set_ylabel('True Positive Rate')
ax[0,1].set_title('(b)'); ax[0,1].legend(loc='lower right')

ConfusionMatrixDisplay(confusion_matrix=cm_sr,
                       display_labels=['Normal','Pneumonia']
    ).plot(cmap='Greens', ax=ax[1,0], colorbar=False)
ax[1,0].set_title('(c) Symbolic Classifier')

ax[1,1].plot(sc_n[0], sc_n[1], color='royalblue',
             label=f'Normal (AUC = {sc_n[2]:.2f})')
ax[1,1].plot(sc_p[0], sc_p[1], color='darkorange',
             label=f'Pneumonia (AUC = {sc_p[2]:.2f})')
ax[1,1].plot([0,1],[0,1],'--', color='black', lw=1)
ax[1,1].set_xlabel('False Positive Rate'); ax[1,1].set_ylabel('True Positive Rate')
ax[1,1].set_title('(d)'); ax[1,1].legend(loc='lower right')

plt.tight_layout()
plt.show()